<a href="https://colab.research.google.com/github/plavebo/SafeAI/blob/main/__v0_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


### 구글드라이브에 있는 pth파일 다운로드 후 연동

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

WEIGHT_ROOT = '/content/drive/MyDrive/안인지'

for f in sorted(os.listdir(WEIGHT_ROOT)):
    print(f)

arc_r100_fp16_backbone.pth
arc_r18_fp16_backbone.pth
arc_r34_fp16_backbone.pth
arc_r50_fp16_backbone.pth
cos_r100_fp16_backbone.pth
cos_r18_fp16_backbone.pth
cos_r34_fp16_backbone.pth
cos_r50_fp16_backbone.pth


### 모델구축시작..

In [ ]:
import sys

!git clone --depth 1 https://github.com/deepinsight/insightface.git /content/insightface
#깃허브에 있는 insightface코드를 colab 서버에 복사해옴

sys.path.insert(0, '/content/insightface/recognition/arcface_torch')
#이 링크에서도 모듈 찾아보라고 알려주는 코드(?)

!pip install -q timm
#timm이란 파이토치 이미지모델을 사용하고 있어서 설치해야함

fatal: destination path '/content/insightface' already exists and is not an empty directory.


In [ ]:
#사용할 모든 라이브러리 가져오기
import torch
import torch.nn as nn
import numpy as np
import torchvision.transforms as T
from pathlib import Path
from PIL import Image
from backbones import iresnet18, iresnet34, iresnet50, iresnet100

# 클론한 insightface코드에서 iResNet모델 구조 가져오기

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'디바이스: {device}')  # cuda 나와야 정상


디바이스: cuda


In [ ]:

#depth이름으로 모델 함수를 찾는 딕셔너리임 ex. BACKBONE_FN['r50']이면 iresnet50 함수 반환함
BACKBONE_FN = {
    'r18':  iresnet18,
    'r34':  iresnet34,
    'r50':  iresnet50,
    'r100': iresnet100,
}

#파일명을 보고 어떤 depth인 지를 알아나는 딕셔너리임
FILE_TO_DEPTH = {
    'arc_r18_fp16_backbone.pth':  'r18',
    'arc_r34_fp16_backbone.pth':  'r34',
    'arc_r50_fp16_backbone.pth':  'r50',
    'arc_r100_fp16_backbone.pth': 'r100',
    'cos_r18_fp16_backbone.pth':  'r18',
    'cos_r34_fp16_backbone.pth':  'r34',
    'cos_r50_fp16_backbone.pth':  'r50',
    'cos_r100_fp16_backbone.pth': 'r100',
}

#load_model 함수
def load_model(filename):
    depth = FILE_TO_DEPTH[filename] #파일명에서 depth를 찾는다
    model = BACKBONE_FN[depth](fp16=True).to(device).eval() #모델 구조를 생성한다
    path = f'{WEIGHT_ROOT}/{filename}' #가중치 경로
    sd = torch.load(path, map_location=device) #가중치 로드
    if any(k.startswith('module.') for k in sd): #module. 제거하기
        sd = {k.replace('module.', ''): v for k, v in sd.items()}
    model.load_state_dict(sd, strict=False) #모델에 가중치 주입하기
    return model


### LFW 평가

#### 예은_이해용 메모
LFW: 얼굴 인식 모델 성능 측정하는 벤치마크임

만약에 6000쌍의 얼굴 사진으로 테스트한다면 3000장은 같은 사람 나머지는 다른사람일 것임
모델한테 두 사진 같은 사람이냐 물어보고 맞으면 정답 틀리면 오답이 나오겠지? 여기서 정확도가 몇 %맞췄는 지임

그래서 8개 모델을 순서대로 LFW 돌려보고 정확도를 비교해서 젤 높은 걸 우리가 써야할듯하다..?

In [ ]:
#LFW 다운로드하기
import kagglehub
import pandas as pd

path = kagglehub.dataset_download("jessicali9530/lfw-dataset")
print("Path:", path)

LFW_PATH = f'{path}/lfw-deepfunneled/lfw-deepfunneled'
KAGGLE_PATH = path

print(f'인물 수: {len(os.listdir(LFW_PATH))}')

Using Colab cache for faster access to the 'lfw-dataset' dataset.
Path: /kaggle/input/lfw-dataset
인물 수: 5749


#### 캐글에 있는 LFW 다운로드해서 씀
: https://www.kaggle.com/datasets/jessicali9530/lfw-dataset

#### 얼굴크롭 전처리(26.05.10 수정)

In [ ]:
!pip install insightface onnxruntime-gpu

In [ ]:
import glob, cv2
from insightface.app import FaceAnalysis
from insightface.utils import face_align

OUTPUT_DIR = '/content/lfw_cropped'
os.makedirs(OUTPUT_DIR, exist_ok=True)

app = FaceAnalysis(name='buffalo_l')
app.prepare(ctx_id=0, det_size=(640, 640))

image_paths = sorted(set(
    glob.glob(os.path.join(LFW_PATH, '**', '*.jpg'), recursive=True)
))
print(f'총 {len(image_paths)}개 이미지 전처리 시작...')

failed = 0
for img_path in image_paths:
    img = cv2.imread(img_path)
    if img is None:
        failed += 1
        continue

    faces = app.get(img)
    if len(faces) == 0:
        failed += 1
        continue

    rel_path = os.path.relpath(img_path, LFW_PATH) #폴더구조 유지
    save_path = os.path.join(OUTPUT_DIR, rel_path)
    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    aimg = face_align.norm_crop(img, landmark=faces[0].kps, image_size=112)
    cv2.imwrite(save_path, aimg)

print(f'✅ 전처리 완료! (실패: {failed}개)')
LFW_PATH = OUTPUT_DIR
print(f'LFW_PATH → {LFW_PATH}')

download_path: /root/.insightface/models/buffalo_l


100%|██████████| 281857/281857 [00:07<00:00, 36098.59KB/s]


Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}, 'CUDAExecutionProvider': {'sdpa_kernel': '0', 'use_tf32': '1', 'fuse_conv_bias': '0', 'prefer_nhwc': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_enable': '0', 'use_ep_level_unified_stream': '0', 'device_id': '0', 'has_user_compute_stream': '0', 'gpu_external_empty_cache': '0', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'cudnn_conv1d_pad_to_nc1d': '0', 'gpu_mem_limit': '18446744073709551615', 'gpu_external_alloc': '0', 'gpu_external_free': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'do_copy_in_default_stream': '1', 'enable_cuda_graph': '0', 'user_compute_stream': '0', 'cudnn_conv_use_max_workspace': '1'}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with o

#### pairs 만들기

In [ ]:
#kaggle에서 pair.csv파일 불러오기

match    = pd.read_csv(f'{KAGGLE_PATH}/matchpairsDevTrain.csv') #match파일에 같은 사람 쌍 목록 있음
mismatch = pd.read_csv(f'{KAGGLE_PATH}/mismatchpairsDevTrain.csv') #mismatch파일에 다른 사람 쌍 목록 있음

def make_pairs(match, mismatch, lfw_path):
    pairs = []
    lfw_path = Path(lfw_path) #문자열 경로를 path 객체로 변환해서 경로 연산을 편하게 함

    def img_path(name, idx):
      #이름, 번호로 실제 이미지 파일 경로 만들기
        return lfw_path / name / f'{name}_{int(idx):04d}.jpg'

    #같은 사람 쌍 처리하기
    for _, row in match.iterrows():
        p1 = img_path(row['name'], row['imagenum1']) #첫번째 이미지 경로
        p2 = img_path(row['name'], row['imagenum2']) #두번째 이미지 경로
        if p1.exists() and p2.exists(): #두 파일이 실제로 모두 존재한다면
            pairs.append((str(p1), str(p2), 1)) #label=1로 해라(같은사람이다)

    #다른 사람 쌍 처리하기
    for _, row in mismatch.iterrows():
        p1 = img_path(row['name'], row['imagenum1']) #첫 번째 이미지 경로
        p2 = img_path(row['name.1'], row['imagenum2']) #두 번째 이미지 경로
        if p1.exists() and p2.exists(): #두 파일이 실제로 존재할 때만
            pairs.append((str(p1), str(p2), 0)) #label=0로 해라(다른사람이다)

    #결과 출력하기
    same = sum(1 for *_, l in pairs if l == 1) #label=1 개수 세기
    diff = sum(1 for *_, l in pairs if l == 0) #label=0 개수 세기
    print(f'총 pairs: {len(pairs)}쌍 (같은사람={same}, 다른사람={diff})')
    return pairs #완성된 pairs 리스트 반환하기

#함수 실행해서 pairs 리스트 생성하기
pairs  = make_pairs(match, mismatch, LFW_PATH)

#pairs=[(경로1,경로2,정답),(경로1,경로2,정답),등등] 이걸 이미지1,이미지2,정답끼리 따로 분리함
imgs1  = [p[0] for p in pairs] #첫번째 이미지 경로만 모음
imgs2  = [p[1] for p in pairs] #두번째 이미지 경로만 모음
labels = np.array([p[2] for p in pairs]) #정답(0/1)만 모음, 나중에 성능평가할 때 연산하기 쉽게 np.array로 변환함

총 pairs: 2188쌍 (같은사람=1093, 다른사람=1095)


#### match, mismatch파일로 pairs를 만든다

### 성능 평가

In [ ]:
#ArcFace 표준처리 파이프라인
#이미지를 모델에 넣기 전에 항상 3단계 이렇게 적어야 함

TRANSFORM = T.Compose([
    T.Resize((112, 112)), #이미지 크기를 112x112px로 고정
    T.ToTensor(), #이미지(PIL)를 숫자행렬(Tensor)로 변환하기
    T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]), #픽셀값 정규화 -1~1로 변환함(?)->모델이 이 범위로 학습했기 때문임
])

@torch.no_grad() #추론할 때 메모리 절약위해 기울기 계산을 안함

def extract_batch(model, img_paths, batch_size=64):
  #이미지 경로 리스트를 받아서 512개 숫자 행렬로 변환함

    all_embs = []

    for i in range(0, len(img_paths), batch_size):

        batch = img_paths[i:i+batch_size]

        tensors = [TRANSFORM(Image.open(p).convert('RGB')) for p in batch]
        #이미지 파일 열고 전처리 적용하면 각 이미지를 열어서 RGB로 변환하고 Transform적용해줌

        x = torch.stack(tensors).to(device).half() #FP32를 FP16로 변환함(GPU에서 쓰려고)
        emb = torch.nn.functional.normalize(model(x), dim=1) #모델에 이미지 넣어서 임베딩 추출함->(64,512)
        #그 다음에 임베딩을 L2 저유화해서 길이를 1로 맞춤->코사인 유사도 계산 편해지기 때문임(?)

        all_embs.append(emb.cpu().float().numpy())
        #CPU로 이동해서 FP16에서 FP32로 변환하고 넘파이로 바꿈

    return np.vstack(all_embs)
    #64장씩 나눠서 처리한 결과 하나로 합침

def evaluate(model, imgs1, imgs2, labels):
    print('  임베딩 추출 중...')
    embs1 = extract_batch(model, imgs1) #첫번째 이미지를 임베딩 추출 ->(2200,512)
    embs2 = extract_batch(model, imgs2) #두번째 이미지를 임베딩 추출 ->(2200,512)

    scores = np.sum(embs1 * embs2, axis=1)
    #두 임베딩의 코사인 유사도를 계산함
    #L2 정규화된 벡터끼리 곱하면 코사인 유사도가 됨
    #결과: (2200, )배열이되고 각 쌍의 유사도 점수를 의미함
      #유사도가 높으면(1에 가까우면) 같은사람인 것임

    best_acc, best_thresh = 0, 0

    for thresh in np.arange(-1.0, 1.0, 0.01):
        preds = (scores >= thresh).astype(int)
        acc = np.mean(preds == labels)

        if acc > best_acc: #더 높은 정확도가 나오면 업데이트
            best_acc, best_thresh = acc, thresh

    return best_acc, best_thresh
    #최고 정확도, 그때의 threshold

#### 모델 순서대로 평가함

In [ ]:
results = {}

for filename in sorted(FILE_TO_DEPTH.keys()):
    print(f'\n{"="*50}') #구분선
    print(f'{filename}') #평가 중인 모델명 출력
    print(f'{"="*50}') #구분선

    model = load_model(filename) #가중치 파일 로드

    acc, thresh = evaluate(model, imgs1, imgs2, labels)
    results[filename] = {'accuracy': acc, 'threshold': thresh}

    print(f'  정확도: {acc*100:.2f}%')
    print(f'  최적 threshold: {thresh:.3f}')

    del model
    torch.cuda.empty_cache()

# 최종 결과
print('\n\n 최종 결과 (정확도 순)')
print('='*50)
for name, r in sorted(results.items(), key=lambda x: x[1]['accuracy'], reverse=True): #정확도가 높은 순으로
    print(f'{r["accuracy"]*100:.2f}%  {name}')


arc_r100_fp16_backbone.pth
  임베딩 추출 중...


/content/insightface/recognition/arcface_torch/backbones/iresnet.py:149: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self.fp16):


  정확도: 98.77%
  최적 threshold: 0.290

arc_r18_fp16_backbone.pth
  임베딩 추출 중...
  정확도: 98.77%
  최적 threshold: 0.240

arc_r34_fp16_backbone.pth
  임베딩 추출 중...
  정확도: 98.67%
  최적 threshold: 0.210

arc_r50_fp16_backbone.pth
  임베딩 추출 중...
  정확도: 98.77%
  최적 threshold: 0.210

cos_r100_fp16_backbone.pth
  임베딩 추출 중...
  정확도: 98.77%
  최적 threshold: 0.230

cos_r18_fp16_backbone.pth
  임베딩 추출 중...
  정확도: 98.77%
  최적 threshold: 0.260

cos_r34_fp16_backbone.pth
  임베딩 추출 중...
  정확도: 98.77%
  최적 threshold: 0.260

cos_r50_fp16_backbone.pth
  임베딩 추출 중...
  정확도: 98.77%
  최적 threshold: 0.210


 최종 결과 (정확도 순)
98.77%  arc_r100_fp16_backbone.pth
98.77%  arc_r18_fp16_backbone.pth
98.77%  arc_r50_fp16_backbone.pth
98.77%  cos_r100_fp16_backbone.pth
98.77%  cos_r18_fp16_backbone.pth
98.77%  cos_r34_fp16_backbone.pth
98.77%  cos_r50_fp16_backbone.pth
98.67%  arc_r34_fp16_backbone.pth


#### 전처리한 코드로 돌린 결과(26.05.10)
전처리한 코드로돌리니 98.7까지 올랐는데 8개중 7쌍이 98.77이나옴

테스트 쌍이 2200쌍이라 차이가 너무 작아 구분이 안됨 더 많은 쌍으로 테스트해야 차이가 드러남

#### 해결방안(?)
LFW는 평가하기위해서 쓰는거니까 LFW내의 Train,Test셋을 합쳐서 더 많은 쌍으로 만들어 평가해보고자함

In [ ]:
#Train+Test합치기
match = pd.concat([
    pd.read_csv(f'{KAGGLE_PATH}/matchpairsDevTrain.csv'),
    pd.read_csv(f'{KAGGLE_PATH}/matchpairsDevTest.csv'),
])
mismatch = pd.concat([
    pd.read_csv(f'{KAGGLE_PATH}/mismatchpairsDevTrain.csv'),
    pd.read_csv(f'{KAGGLE_PATH}/mismatchpairsDevTest.csv'),
])
print(f'match: {len(match)}쌍, mismatch: {len(mismatch)}쌍')

# pairs 다시 만들기
pairs  = make_pairs(match, mismatch, LFW_PATH)
imgs1  = [p[0] for p in pairs]
imgs2  = [p[1] for p in pairs]
labels = np.array([p[2] for p in pairs])

match: 1600쌍, mismatch: 1600쌍
총 pairs: 3181쌍 (같은사람=1591, 다른사람=1590)


#### 결과
##### 아직도 모델 간 차이가 0.1%정도라 비슷한 성능임
##### 최종 결과 (정확도 순)
==================================================
98.55%  arc_r18_fp16_backbone.pth
98.55%  arc_r50_fp16_backbone.pth
98.55%  cos_r100_fp16_backbone.pth
98.52%  arc_r100_fp16_backbone.pth
98.52%  cos_r18_fp16_backbone.pth
98.52%  cos_r34_fp16_backbone.pth
98.52%  cos_r50_fp16_backbone.pth
98.46%  arc_r34_fp16_backbone.pth

가중치 결정 기준
속도가 중요하면 arc_r18
성능이 중요하면 cos_r100
균형적인건 arc_50 이라고함

이 상황에선 cos_r100_fp16이 가장 나을 것 같긴함
성능 98.55%나왔음
Glint360k라서 17만장으로 학습했음
iResNet-100이라서 가장 깊은 구조임